# 03 — In-domain Evaluation (M5)

Evaluate the trained baseline on the **held-out controlled (PlantVillage) test
split** — the same leaf-grouped split built in M3, which the model never saw.

Reports **accuracy, per-class precision/recall/F1, macro-F1, the confusion matrix,
and the per-class false-negative rate** (FN-rate = `1 - recall`: the fraction of
truly-diseased leaves the model *missed* — the number that matters most for
disease detection).

> **This notebook runs locally on CPU** — it's inference, not training. It needs:
> - `outputs/checkpoints/best.pt` (your Colab-trained model from notebook 02), and
> - `data/processed/plantvillage_splits.csv` + the PlantVillage images.
>
> ⚠️ **Invariant #5:** these are **controlled-domain numbers only**. They are not
> the headline and mean nothing on their own — they exist to be compared against
> the M6 field (CD&S) results. The domain gap is the actual finding.

## 1. Setup

In [ ]:
import os
import sys
from pathlib import Path

# Local Jupyter starts in notebooks/ — step up to the repo root.
if Path.cwd().name == "notebooks":
    os.chdir("..")
PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

ckpt = PROJECT_ROOT / "outputs/checkpoints/best.pt"
splits = PROJECT_ROOT / "data/processed/plantvillage_splits.csv"
print("Project root:", PROJECT_ROOT)
print("checkpoint present:", ckpt.exists())
print("splits present:    ", splits.exists())
if not ckpt.exists():
    print("\n-> Train on Colab (notebook 02) and place best.pt at outputs/checkpoints/best.pt.")
if not splits.exists():
    print("\n-> Build splits: python -m src.maize_detection.data")

## 2. Run the evaluation

`evaluate.main()` loads `best.pt`, runs the **test** split through the model using
the pretrained weights' own eval transforms (invariant #4), prints the report, and
writes `outputs/metrics_in_domain.json` + the confusion-matrix figure. The printed
report ends with the **invariant #5 reminder** and the **common_rust leakage
caveat** so they can't be dropped.

In [ ]:
from src.maize_detection.evaluate import main as run_eval
run_eval()

## 3. Confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig_path = PROJECT_ROOT / "outputs/figures/confusion_matrix_in_domain.png"
plt.figure(figsize=(7, 6))
plt.imshow(Image.open(fig_path))
plt.axis("off")
plt.show()

## 4. Reading these numbers

- **Per-class FN-rate** is the headline disease-detection number: a high FN-rate
  means the model silently misses that disease. Watch **gray_leaf_spot** (the
  minority class) and **northern_leaf_blight** in particular.
- **common_rust** recall carries the documented residual leakage risk
  (`data.COMMON_RUST_LEAKAGE_CAVEAT`) — treat a near-perfect rust score with
  skepticism; it may be optimistic.
- These are **controlled** images (uniform 256×256 lab crops). The next milestone
  (**M6**) runs the *same* model on **field** CD&S photos and reports the two side
  by side. Expect a large drop — that gap is the project's actual result.